<a href="https://colab.research.google.com/github/victorortizrebolloso-cmyk/ML-ABT-Design/blob/Code/ABT_28_octubre.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
import pandas as pd
import numpy as np

In [19]:
# Load the datasets
basic = pd.read_csv('/content/risk_train_BASIC_DATA_26_27.txt', sep='\t', encoding='cp1252')
comm = pd.read_excel('/content/risk_train_COMM_DATA_26_27.xlsx')
debit = pd.read_excel('/content/DEBIT_CARD_DB_26_27.xlsx')
credit = pd.read_excel('/content/CREDIT_CARD_DB_26_27.xlsx')

In [20]:

# 1. Basic Data Inspection
print("--- BASIC DATA ---")
print("Columns in Basic (Notice 'ID' vs 'ID_OFF'):", basic.columns.tolist())
print("Missing AGE (#N/D):", (basic['AGE'] == '#N/D').sum())
print("Missing INCOME (#N/D):", (basic['INCOME'] == '#N/D').sum())
print("Unique GENDERs:", basic['GENDER'].unique())
print("SOURCE_SYSTEMs:", basic['SOURCE_SYSTEM'].unique())
print("Sample INCOME format:", basic['INCOME'].head(3).tolist())
print("Number of duplicated rows:", basic.duplicated().sum())

# 2. Commercial Data Inspection
print("\n--- COMM DATA ---")
print("Columns in Comercial (Notice 'ID' vs 'ID_OFF'):", comm.columns.tolist())
print("Missing #OFFICE:", comm['#OFFICE'].isnull().sum())
print("Unique MORTGAGE values:", comm['MORTGAGE'].unique())
print("CARD_EXPORT counts:\n", comm['CARD_EXPORT'].value_counts().to_dict())
print("Number of duplicated rows:", comm.duplicated().sum())


# 3. Debit Card Data Inspection
print("\n--- DEBIT DATA ---")
print("Columns in Debit Card (Notice 'ID' vs 'ID_OFF'):", debit.columns.tolist())
print("Total rows:", len(debit), "| Unique customers:", debit['ID_OFF'].nunique())
print("Unique ACTIVE values:", debit['ACTIVE'].unique())
print("Number of duplicated rows:", debit.duplicated().sum())


# 4. Credit Card Data Inspection
print("\n--- CREDIT DATA ---")
print("Columns in Credit (Notice 'ID' vs 'ID_OFF'):", credit.columns.tolist())
print("Sample credit ID_OFF (Notice 'REF/' prefix):", credit['ID_OFF'].head(3).tolist())
print("Duplicate card records:", credit['CREDIT_CARD_NUM'].duplicated().sum())
print("Purchases < 0 (2021):", (credit['CREDIT_USE_2021'] < 0).sum())
print("Purchases < 0 (2022):", (credit['CREDIT_USE_2022'] < 0).sum())
print("Purchases > 25000 (2021):", (credit['CREDIT_USE_2021'] > 25000).sum())
print("Purchases > 25000 (2022):", (credit['CREDIT_USE_2022'] > 25000).sum())
print("Number of duplicated rows:", credit.duplicated().sum())




--- BASIC DATA ---
Columns in Basic (Notice 'ID' vs 'ID_OFF'): ['ID', 'MARITAL_STATUS', 'AGE', 'MARITAL_STATUS.1', 'NUMKIDS', 'HOUSEHOLD_MEMBERS', 'GENDER', 'INCOME', 'CUSTOMER_SINCE', 'SOURCE_SYSTEM']
Missing AGE (#N/D): 40
Missing INCOME (#N/D): 170
Unique GENDERs: ['male' 'female' 'FEMALE ' ' fem ']
SOURCE_SYSTEMs: ['M' 'L']
Sample INCOME format: [' 4.749,94 € ', ' 6.545,39 € ', ' 2.485,18 € ']
Number of duplicated rows: 0

--- COMM DATA ---
Columns in Comercial (Notice 'ID' vs 'ID_OFF'): ['ID_OFF', 'HOWPAID', 'MORTGAGE', 'LOANS', 'RISK', '#OFFICE', 'PRODUCT', 'DATE_CREDIT_OP', 'CARD_EXPORT']
Missing #OFFICE: 143
Unique MORTGAGE values: ['y' 'n' 'yes']
CARD_EXPORT counts:
 {'C': 19306, 'X': 120}
Number of duplicated rows: 0

--- DEBIT DATA ---
Columns in Debit Card (Notice 'ID' vs 'ID_OFF'): ['ID_OFF', 'DEBIT_CARD_NUM', 'ACTIVE']
Total rows: 44141 | Unique customers: 18886
Unique ACTIVE values: ['YES' 'NO' ' Y ' 'N']
Number of duplicated rows: 0

--- CREDIT DATA ---
Columns in Credi

In [21]:
debit.head()

,ID_OFF,DEBIT_CARD_NUM,ACTIVE
0,46911_48,7988079626878,YES
1,47236_53,7994070642624,NO
2,45517_88,7139952033170,YES
3,43523_55,7214918140884,YES
4,41876_55,7382421478789,YES


In [22]:
# Explicitly normalize key name in basic data
basic = basic.rename(columns={'ID': 'ID_OFF'})

# Filter commercial universe ('C' cards only)
comm_filtered = comm[comm['CARD_EXPORT'] == 'C'].copy()

# Aggregate Debit data (Card-level -> Customer-level count)
debit_agg = debit.groupby('ID_OFF').size().reset_index(name='TOTAL_DEBIT_CARDS')

# Clean & Aggregate Credit data (Deduplicate latest version per card, sum purchases)
credit['ID_OFF'] = credit['ID_OFF'].str.replace('REF/', '', regex=False)
credit_agg = credit.sort_values('UPDATED_AT').drop_duplicates(subset=['ID_OFF', 'CREDIT_CARD_NUM'], keep='last') \
                   .groupby('ID_OFF')[['CREDIT_USE_2021', 'CREDIT_USE_2022']].sum().reset_index()

# Integrate ABT via sequential Left Joins
abt = comm_filtered.merge(basic, on='ID_OFF', how='left') \
                   .merge(debit_agg, on='ID_OFF', how='left') \
                   .merge(credit_agg, on='ID_OFF', how='left')

# Fill missing card values with true zeros for valid customers
card_cols = ['TOTAL_DEBIT_CARDS', 'CREDIT_USE_2021', 'CREDIT_USE_2022']
abt[card_cols] = abt[card_cols].fillna(0)

print(abt.info())
print("Total rows:", len(abt))
print("Duplicated rows:", abt.duplicated().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19306 entries, 0 to 19305
Data columns (total 21 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   ID_OFF             19306 non-null  object        
 1   HOWPAID            19306 non-null  object        
 2   MORTGAGE           19306 non-null  object        
 3   LOANS              19306 non-null  int64         
 4   RISK               19306 non-null  object        
 5   #OFFICE            19163 non-null  float64       
 6   PRODUCT            19306 non-null  object        
 7   DATE_CREDIT_OP     19306 non-null  datetime64[ns]
 8   CARD_EXPORT        19306 non-null  object        
 9   MARITAL_STATUS     19275 non-null  object        
 10  AGE                19275 non-null  object        
 11  MARITAL_STATUS.1   19275 non-null  float64       
 12  NUMKIDS            19275 non-null  float64       
 13  HOUSEHOLD_MEMBERS  19275 non-null  float64       
 14  GENDER

In [23]:
abt

,ID_OFF,HOWPAID,MORTGAGE,LOANS,RISK,#OFFICE,PRODUCT,DATE_CREDIT_OP,CARD_EXPORT,MARITAL_STATUS,...,MARITAL_STATUS.1,NUMKIDS,HOUSEHOLD_MEMBERS,GENDER,INCOME,CUSTOMER_SINCE,SOURCE_SYSTEM,TOTAL_DEBIT_CARDS,CREDIT_USE_2021,CREDIT_USE_2022
0,30571_55,monthly,y,0,bad loss,55.0,CLASSIC,2023-07-19,C,divsepwid,...,1.0,1.0,3.0,male,"47.881,13 €",2021-07-11,L,2.0,4483.08,3725.37
1,36153_11,weekly,y,1,good risk,11.0,BASIC,2023-04-29,C,single,...,0.0,0.0,1.0,female,"20.040,77 €",2021-07-29,L,2.0,0.00,0.00
2,38055_20,weekly,n,3,good risk,20.0,CLASSIC,2023-04-24,C,single,...,0.0,0.0,1.0,fem,"3.164,03 €",2016-09-11,M,3.0,14107.52,13807.14
3,39623_50,monthly,y,1,good risk,50.0,EXCEL,2023-10-21,C,single,...,0.0,0.0,1.0,female,"3.498,58 €",2018-07-29,M,2.0,881.70,523.14
4,32974_13,weekly,y,1,bad profit,13.0,classic,2023-10-18,C,single,...,0.0,0.0,1.0,female,"49.294,77 €",2010-09-30,L,2.0,7354.00,6936.50
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19301,30607_21,monthly,y,0,bad loss,21.0,CLASSIC,2023-11-21,C,married,...,1.0,0.0,2.0,female,"90.425,16 €",2007-10-18,L,1.0,78.90,44.11
19302,30031_62,monthly,y,0,bad loss,62.0,CLASSIC,2023-09-17,C,married,...,1.0,3.0,6.0,male,"62.669,33 €",2021-10-02,L,2.0,3732.93,2763.33
19303,30829_24,weekly,y,2,bad profit,24.0,PERSONAL,2023-08-13,C,single,...,0.0,0.0,1.0,female,"32.699,70 €",2021-01-17,L,5.0,12325.73,9222.99
19304,33738_21,monthly,y,1,bad profit,21.0,PERSONAL,2023-08-08,C,married,...,1.0,0.0,2.0,female,"65.958,64 €",2001-05-09,L,2.0,4853.97,4365.99


In [24]:
# 1. Drop redundancies: MARITAL_STATUS.1 is a duplicate; CARD_EXPORT is exclusively 'C' now.
abt = abt.drop(columns=['MARITAL_STATUS.1', 'CARD_EXPORT'])

# 2. Format Categoricals: Standardize GENDER to unify text; binarize MORTGAGE for modeling.
abt['GENDER'] = abt['GENDER'].astype(str).str.lower().str.strip().replace({'fem': 'female'})
abt['MORTGAGE'] = abt['MORTGAGE'].astype(str).str.lower().map({'y': 1, 'yes': 1, 'n': 0, 'no': 0})

# 3. Parse Numerics: Coerce AGE to numeric (safely converting '#N/D' string to NaN).
abt['AGE'] = pd.to_numeric(abt['AGE'], errors='coerce')

# Strip currency symbols and European formatting from INCOME, convert to float.
abt['INCOME'] = abt['INCOME'].astype(str).str.replace('€', '', regex=False) \
                             .str.replace('.', '', regex=False) \
                             .str.replace(',', '.', regex=False)
abt['INCOME'] = pd.to_numeric(abt['INCOME'], errors='coerce')

# 4. Apply Business Logic: Annualize income ONLY for customers explicitly marked as Monthly ('M').
abt.loc[abt['SOURCE_SYSTEM'] == 'M', 'INCOME'] *= 12

# Handle Anomalies:
# - Corrupted negative incomes become NaN to be imputed.
# - Negative credit is a system error (NaN). Limit max at 25,000 ceiling.

abt.loc[abt['INCOME'] < 0, 'INCOME'] = np.nan
abt.loc[abt['CREDIT_USE_2021'] < 0, 'CREDIT_USE_2021'] = np.nan
abt.loc[abt['CREDIT_USE_2022'] < 0, 'CREDIT_USE_2022'] = np.nan
abt.loc[abt['CREDIT_USE_2021'] > 25000, 'CREDIT_USE_2021'] = np.nan
abt.loc[abt['CREDIT_USE_2022'] > 25000, 'CREDIT_USE_2022'] = np.nan


# 5. Impute Missing Values: Use robust median to maintain distribution shape; flag unknown #OFFICE as -1 to retain records.
abt['AGE'] = abt['AGE'].fillna(abt['AGE'].median())
abt['INCOME'] = abt['INCOME'].fillna(abt['INCOME'].median())
abt['#OFFICE'] = abt['#OFFICE'].fillna(-1)

# Verify cleaning results
print(abt.info())
print("Nulls remaining:\n", abt.isnull().sum()[abt.isnull().sum() > 0])

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19306 entries, 0 to 19305
Data columns (total 19 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   ID_OFF             19306 non-null  object        
 1   HOWPAID            19306 non-null  object        
 2   MORTGAGE           19306 non-null  int64         
 3   LOANS              19306 non-null  int64         
 4   RISK               19306 non-null  object        
 5   #OFFICE            19306 non-null  float64       
 6   PRODUCT            19306 non-null  object        
 7   DATE_CREDIT_OP     19306 non-null  datetime64[ns]
 8   MARITAL_STATUS     19275 non-null  object        
 9   AGE                19306 non-null  float64       
 10  NUMKIDS            19275 non-null  float64       
 11  HOUSEHOLD_MEMBERS  19275 non-null  float64       
 12  GENDER             19306 non-null  object        
 13  INCOME             19306 non-null  float64       
 14  CUSTOM

In [25]:
abt

,ID_OFF,HOWPAID,MORTGAGE,LOANS,RISK,#OFFICE,PRODUCT,DATE_CREDIT_OP,MARITAL_STATUS,AGE,NUMKIDS,HOUSEHOLD_MEMBERS,GENDER,INCOME,CUSTOMER_SINCE,SOURCE_SYSTEM,TOTAL_DEBIT_CARDS,CREDIT_USE_2021,CREDIT_USE_2022
0,30571_55,monthly,1,0,bad loss,55.0,CLASSIC,2023-07-19,divsepwid,35.0,1.0,3.0,male,47881.13,2021-07-11,L,2.0,4483.08,3725.37
1,36153_11,weekly,1,1,good risk,11.0,BASIC,2023-04-29,single,27.0,0.0,1.0,female,20040.77,2021-07-29,L,2.0,0.00,0.00
2,38055_20,weekly,0,3,good risk,20.0,CLASSIC,2023-04-24,single,40.0,0.0,1.0,female,37968.36,2016-09-11,M,3.0,14107.52,13807.14
3,39623_50,monthly,1,1,good risk,50.0,EXCEL,2023-10-21,single,42.0,0.0,1.0,female,41982.96,2018-07-29,M,2.0,881.70,523.14
4,32974_13,weekly,1,1,bad profit,13.0,classic,2023-10-18,single,40.0,0.0,1.0,female,49294.77,2010-09-30,L,2.0,7354.00,6936.50
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19301,30607_21,monthly,1,0,bad loss,21.0,CLASSIC,2023-11-21,married,43.0,0.0,2.0,female,90425.16,2007-10-18,L,1.0,78.90,44.11
19302,30031_62,monthly,1,0,bad loss,62.0,CLASSIC,2023-09-17,married,36.0,3.0,6.0,male,62669.33,2021-10-02,L,2.0,3732.93,2763.33
19303,30829_24,weekly,1,2,bad profit,24.0,PERSONAL,2023-08-13,single,31.0,0.0,1.0,female,32699.70,2021-01-17,L,5.0,12325.73,9222.99
19304,33738_21,monthly,1,1,bad profit,21.0,PERSONAL,2023-08-08,married,46.0,0.0,2.0,female,65958.64,2001-05-09,L,2.0,4853.97,4365.99


In [26]:


# Combination 1: Business Logic & Interpretability
# Added missing minus (-) signs for subtraction
abt['TENURE_YEARS'] = (pd.to_datetime('2023-01-01') - pd.to_datetime(abt['CUSTOMER_SINCE'])).dt.days / 365.25
abt['INCOME_PER_CAPITA'] = abt['INCOME'] / abt['HOUSEHOLD_MEMBERS'].replace(0, 1)
abt['CREDIT_TREND'] = abt['CREDIT_USE_2022'] - abt['CREDIT_USE_2021']

# Combination 2: Financial Stress & Dependency Ratios
abt['UTILIZATION_PROXY'] = abt['CREDIT_USE_2022'] / abt['INCOME'].replace(0, np.nan)
abt['DEPENDENT_BURDEN'] = abt['NUMKIDS'] / abt['HOUSEHOLD_MEMBERS'].replace(0, 1)

abt['TENURE_MONTHS'] = np.floor(abt['TENURE_YEARS'] * 12)

# Combination 3: Categorical Discretization (Preserving originals)
abt['CREDIT_TREND_CAT'] = np.select(
    [abt['CREDIT_TREND'] > 0, abt['CREDIT_TREND'] < 0],
    ['Increased', 'Decreased'],
    default='Inactive'
)

# Display final structure parameters
print("Final ABT shape:", abt.shape)
print(abt[['TENURE_YEARS', 'INCOME_PER_CAPITA', 'UTILIZATION_PROXY']].head())
print(abt[['TENURE_YEARS', 'INCOME_PER_CAPITA', 'UTILIZATION_PROXY']].head())

Final ABT shape: (19306, 26)
   TENURE_YEARS  INCOME_PER_CAPITA  UTILIZATION_PROXY
0      1.475702       15960.376667           0.077805
1      1.426420       20040.770000           0.000000
2      6.305270       37968.360000           0.363649
3      4.427105       41982.960000           0.012461
4     12.254620       49294.770000           0.140715
   TENURE_YEARS  INCOME_PER_CAPITA  UTILIZATION_PROXY
0      1.475702       15960.376667           0.077805
1      1.426420       20040.770000           0.000000
2      6.305270       37968.360000           0.363649
3      4.427105       41982.960000           0.012461
4     12.254620       49294.770000           0.140715


In [27]:
display(abt)

,ID_OFF,HOWPAID,MORTGAGE,LOANS,RISK,#OFFICE,PRODUCT,DATE_CREDIT_OP,MARITAL_STATUS,AGE,...,TOTAL_DEBIT_CARDS,CREDIT_USE_2021,CREDIT_USE_2022,TENURE_YEARS,INCOME_PER_CAPITA,CREDIT_TREND,UTILIZATION_PROXY,DEPENDENT_BURDEN,TENURE_MONTHS,CREDIT_TREND_CAT
0,30571_55,monthly,1,0,bad loss,55.0,CLASSIC,2023-07-19,divsepwid,35.0,...,2.0,4483.08,3725.37,1.475702,15960.376667,-757.71,0.077805,0.333333,17.0,Decreased
1,36153_11,weekly,1,1,good risk,11.0,BASIC,2023-04-29,single,27.0,...,2.0,0.00,0.00,1.426420,20040.770000,0.00,0.000000,0.000000,17.0,Inactive
2,38055_20,weekly,0,3,good risk,20.0,CLASSIC,2023-04-24,single,40.0,...,3.0,14107.52,13807.14,6.305270,37968.360000,-300.38,0.363649,0.000000,75.0,Decreased
3,39623_50,monthly,1,1,good risk,50.0,EXCEL,2023-10-21,single,42.0,...,2.0,881.70,523.14,4.427105,41982.960000,-358.56,0.012461,0.000000,53.0,Decreased
4,32974_13,weekly,1,1,bad profit,13.0,classic,2023-10-18,single,40.0,...,2.0,7354.00,6936.50,12.254620,49294.770000,-417.50,0.140715,0.000000,147.0,Decreased
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19301,30607_21,monthly,1,0,bad loss,21.0,CLASSIC,2023-11-21,married,43.0,...,1.0,78.90,44.11,15.206023,45212.580000,-34.79,0.000488,0.000000,182.0,Decreased
19302,30031_62,monthly,1,0,bad loss,62.0,CLASSIC,2023-09-17,married,36.0,...,2.0,3732.93,2763.33,1.248460,10444.888333,-969.60,0.044094,0.500000,14.0,Decreased
19303,30829_24,weekly,1,2,bad profit,24.0,PERSONAL,2023-08-13,single,31.0,...,5.0,12325.73,9222.99,1.954825,32699.700000,-3102.74,0.282051,0.000000,23.0,Decreased
19304,33738_21,monthly,1,1,bad profit,21.0,PERSONAL,2023-08-08,married,46.0,...,2.0,4853.97,4365.99,21.648186,32979.320000,-487.98,0.066193,0.000000,259.0,Decreased


In [28]:
#ABT_XLS = abt.to_excel('ABT_semi.xlsx', sheet_name='ABT a pulir', index=False)